In [1]:
import sys
import pandas as pd
import sklearn

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

Python: 3.10.20 (main, Jun 11 2026, 15:17:37) [GCC 14.3.0]
Pandas: 1.5.3
Scikit-learn: 1.7.2


In [2]:
import pandas as pd

# Load the Data Quality Agent dataset
df = pd.read_csv("ai_ml_data_quality_dataset.csv")

# Basic dataset inspection
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
print(df["needs_review"].value_counts())

print("\nTarget percentages:")
print(df["needs_review"].value_counts(normalize=True).mul(100).round(2))

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (50000, 8)

Column names:
['task_type', 'missing_value_count', 'duplicate_flag', 'format_error_flag', 'response_length', 'review_score', 'latency_seconds', 'needs_review']

Missing values:
task_type              0
missing_value_count    0
duplicate_flag         0
format_error_flag      0
response_length        0
review_score           0
latency_seconds        0
needs_review           0
dtype: int64

Duplicate rows: 0

Target distribution:
0    44888
1     5112
Name: needs_review, dtype: int64

Target percentages:
0    89.78
1    10.22
Name: needs_review, dtype: float64

First 5 rows:


,task_type,missing_value_count,duplicate_flag,format_error_flag,response_length,review_score,latency_seconds,needs_review
0,Data Labeling,0,0,1,150.07,75.15,69.08,0
1,Safety Review,0,1,0,98.37,76.21,78.18,0
2,Text Evaluation,3,0,0,208.74,86.67,52.63,0
3,Data Labeling,1,0,0,131.36,89.42,52.59,0
4,Data Labeling,1,0,0,267.50,86.02,145.91,0


In [3]:
# Prepare data for machine learning

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Separate features and target
X = df.drop("needs_review", axis=1)
y = df["needs_review"]

# Identify feature types
categorical_features = ["task_type"]
numerical_features = [
    "missing_value_count",
    "duplicate_flag",
    "format_error_flag",
    "response_length",
    "review_score",
    "latency_seconds"
]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numerical_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("\nTraining target distribution:")
print(y_train.value_counts())
print("\nTesting target distribution:")
print(y_test.value_counts())

Training records: 40000
Testing records: 10000

Training target distribution:
0    35910
1     4090
Name: needs_review, dtype: int64

Testing target distribution:
0    8978
1    1022
Name: needs_review, dtype: int64


In [4]:
# Train baseline machine learning models

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
}

results = []

# Train and evaluate each model
for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

results_df = pd.DataFrame(results)

print("Baseline Model Results:")
display(results_df.round(4))

Baseline Model Results:


,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.9023,0.6380,0.1018,0.1755
1,Decision Tree,0.8953,0.4429,0.0949,0.1563
2,Random Forest,0.9007,0.6124,0.0773,0.1373


In [5]:
# Optimize models using class weighting

weighted_models = {
    "Logistic Regression (Balanced)": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    
    "Decision Tree (Balanced)": DecisionTreeClassifier(
        max_depth=8,
        class_weight="balanced",
        random_state=42
    ),
    
    "Random Forest (Balanced)": RandomForestClassifier(
        n_estimators=150,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

weighted_results = []

for name, model in weighted_models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)

    weighted_results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

weighted_results_df = pd.DataFrame(weighted_results)

print("Class-Weighted Model Results:")
display(weighted_results_df.round(4))

Class-Weighted Model Results:


,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression (Balanced),0.7199,0.2163,0.6634,0.3262
1,Decision Tree (Balanced),0.7189,0.2055,0.6106,0.3075
2,Random Forest (Balanced),0.7835,0.2443,0.5342,0.3353


In [7]:
import numpy as np
# Optimize the decision threshold for the class-weighted models

from sklearn.metrics import precision_recall_curve

threshold_results = []

for name, model in weighted_models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    # Get probability that each record needs review
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    # Test thresholds from 0.10 to 0.90
    for threshold in np.arange(0.10, 0.91, 0.05):

        predictions = (probabilities >= threshold).astype(int)

        precision = precision_score(
            y_test, predictions, zero_division=0
        )
        recall = recall_score(
            y_test, predictions, zero_division=0
        )
        f1 = f1_score(
            y_test, predictions, zero_division=0
        )
        accuracy = accuracy_score(
            y_test, predictions
        )

        threshold_results.append({
            "Model": name,
            "Threshold": round(threshold, 2),
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1": f1
        })

threshold_df = pd.DataFrame(threshold_results)

# Find the best threshold for each model based on F1
best_thresholds = (
    threshold_df
    .sort_values(["Model", "F1"], ascending=[True, False])
    .groupby("Model")
    .head(1)
    .reset_index(drop=True)
)

print("Best Threshold for Each Model:")
display(best_thresholds.round(4))

Best Threshold for Each Model:


,Model,Threshold,Accuracy,Precision,Recall,F1
0,Decision Tree (Balanced),0.60,0.8136,0.2640,0.4609,0.3357
1,Logistic Regression (Balanced),0.65,0.8370,0.3036,0.4599,0.3658
2,Random Forest (Balanced),0.55,0.8221,0.2751,0.4530,0.3423


In [8]:
# Proper validation split for threshold tuning
from sklearn.model_selection import train_test_split

# Split the training data into training and validation sets
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Final training records:", len(X_train_final))
print("Validation records:", len(X_val))
print("Test records:", len(X_test))

print("\nValidation target distribution:")
print(y_val.value_counts())

Final training records: 32000
Validation records: 8000
Test records: 10000

Validation target distribution:
0    7182
1     818
Name: needs_review, dtype: int64


In [9]:
# Train the balanced Logistic Regression model on the final training set
balanced_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    ))
])

balanced_lr.fit(X_train_final, y_train_final)

# Get probabilities for the validation set
val_probabilities = balanced_lr.predict_proba(X_val)[:, 1]

# Test thresholds from 0.10 to 0.90
validation_threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):
    val_predictions = (val_probabilities >= threshold).astype(int)

    validation_threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(y_val, val_predictions),
        "Precision": precision_score(y_val, val_predictions, zero_division=0),
        "Recall": recall_score(y_val, val_predictions, zero_division=0),
        "F1": f1_score(y_val, val_predictions, zero_division=0)
    })

validation_threshold_df = pd.DataFrame(validation_threshold_results)

# Find the threshold with the highest F1
best_validation_threshold = (
    validation_threshold_df
    .sort_values("F1", ascending=False)
    .iloc[0]
)

print("Best Validation Threshold:")
display(best_validation_threshold.to_frame().T.round(4))

print("\nAll Validation Threshold Results:")
display(validation_threshold_df.round(4))

Best Validation Threshold:


,Threshold,Accuracy,Precision,Recall,F1
12,0.7,0.8676,0.3689,0.4144,0.3903



All Validation Threshold Results:


,Threshold,Accuracy,Precision,Recall,F1
0,0.10,0.1208,0.1040,0.9976,0.1883
1,0.15,0.1844,0.1106,0.9902,0.1989
2,0.20,0.2699,0.1199,0.9682,0.2133
3,0.25,0.3575,0.1314,0.9413,0.2305
4,0.30,0.4492,0.1455,0.8998,0.2504
5,0.35,0.5315,0.1617,0.8557,0.2720
6,0.40,0.6051,0.1785,0.7946,0.2915
7,0.45,0.6696,0.1975,0.7286,0.3108
8,0.50,0.7240,0.2209,0.6724,0.3325
9,0.55,0.7684,0.2439,0.6027,0.3473


In [10]:
# Final evaluation on the untouched test set

# Use the validation-selected threshold
final_threshold = best_validation_threshold["Threshold"]

# Get probabilities for the test set
test_probabilities = balanced_lr.predict_proba(X_test)[:, 1]

# Convert probabilities into final predictions
test_predictions = (test_probabilities >= final_threshold).astype(int)

# Calculate final metrics
final_accuracy = accuracy_score(y_test, test_predictions)
final_precision = precision_score(y_test, test_predictions, zero_division=0)
final_recall = recall_score(y_test, test_predictions, zero_division=0)
final_f1 = f1_score(y_test, test_predictions, zero_division=0)

print("FINAL TEST RESULTS")
print("------------------")
print(f"Threshold: {final_threshold:.2f}")
print(f"Accuracy:  {final_accuracy:.4f}")
print(f"Precision: {final_precision:.4f}")
print(f"Recall:    {final_recall:.4f}")
print(f"F1 Score:  {final_f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    test_predictions,
    target_names=["No Review", "Needs Review"],
    zero_division=0
))

FINAL TEST RESULTS
------------------
Threshold: 0.70
Accuracy:  0.8615
Precision: 0.3390
Recall:    0.3738
F1 Score:  0.3555

Confusion Matrix:


In [11]:
# Import confusion_matrix
from sklearn.metrics import confusion_matrix

# Display the confusion matrix
print("Final Confusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

Final Confusion Matrix:
[[8233  745]
 [ 640  382]]


In [12]:
# Hyperparameter tuning for balanced Logistic Regression

from sklearn.model_selection import GridSearchCV

# Build the Logistic Regression pipeline
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    ))
])

# Hyperparameters to test
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

# Use F1 as the scoring metric because the dataset is imbalanced
grid_search = GridSearchCV(
    lr_pipeline,
    param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

# Train and evaluate the different configurations
grid_search.fit(X_train_final, y_train_final)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation F1:")
print(round(grid_search.best_score_, 4))

Best Parameters:
{'model__C': 100}

Best Cross-Validation F1:
0.334


In [13]:
# Final optimized Logistic Regression model

final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        C=100,
        class_weight="balanced",
        max_iter=1000
    ))
])

# Train the optimized model on the final training data
final_model.fit(X_train_final, y_train_final)

print("Final optimized model trained successfully.")

Final optimized model trained successfully.


In [14]:
# Evaluate the optimized model on the untouched test set

test_probabilities = final_model.predict_proba(X_test)[:, 1]

# Use the threshold selected during validation
final_threshold = 0.70

test_predictions = (test_probabilities >= final_threshold).astype(int)

final_accuracy = accuracy_score(y_test, test_predictions)
final_precision = precision_score(y_test, test_predictions, zero_division=0)
final_recall = recall_score(y_test, test_predictions, zero_division=0)
final_f1 = f1_score(y_test, test_predictions, zero_division=0)

print("FINAL OPTIMIZED TEST RESULTS")
print("----------------------------")
print(f"Threshold: {final_threshold:.2f}")
print(f"Accuracy:  {final_accuracy:.4f}")
print(f"Precision: {final_precision:.4f}")
print(f"Recall:    {final_recall:.4f}")
print(f"F1 Score:  {final_f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

FINAL OPTIMIZED TEST RESULTS
----------------------------
Threshold: 0.70
Accuracy:  0.8616
Precision: 0.3395
Recall:    0.3748
F1 Score:  0.3563

Confusion Matrix:
[[8233  745]
 [ 639  383]]


In [15]:
# Compare model performance across the optimization stages

comparison_results = pd.DataFrame([
    {
        "Stage": "Baseline Logistic Regression",
        "Accuracy": 0.9023,
        "Precision": 0.6380,
        "Recall": 0.1018,
        "F1": 0.1755
    },
    {
        "Stage": "Class-Weighted Logistic Regression",
        "Accuracy": 0.7199,
        "Precision": 0.2163,
        "Recall": 0.6634,
        "F1": 0.3262
    },
    {
        "Stage": "Threshold-Tuned Logistic Regression",
        "Accuracy": 0.8615,
        "Precision": 0.3395,
        "Recall": 0.3748,
        "F1": 0.3563
    },
    {
        "Stage": "Hyperparameter-Tuned Logistic Regression",
        "Accuracy": 0.8615,
        "Precision": 0.3395,
        "Recall": 0.3748,
        "F1": 0.3563
    }
])

print("MODEL PERFORMANCE COMPARISON")
display(comparison_results.round(4))

MODEL PERFORMANCE COMPARISON


,Stage,Accuracy,Precision,Recall,F1
0,Baseline Logistic Regression,0.9023,0.6380,0.1018,0.1755
1,Class-Weighted Logistic Regression,0.7199,0.2163,0.6634,0.3262
2,Threshold-Tuned Logistic Regression,0.8615,0.3395,0.3748,0.3563
3,Hyperparameter-Tuned Logistic Regression,0.8615,0.3395,0.3748,0.3563


In [16]:
# Register the final optimized model in Azure Machine Learning

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes
import joblib

# Save the trained model locally
joblib.dump(final_model, "data_quality_agent_model.pkl")

print("Model saved successfully as data_quality_agent_model.pkl")

Model saved successfully as data_quality_agent_model.pkl


In [17]:
# Connect to the Azure ML workspace

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

# Connect using the current Azure ML workspace configuration
ml_client = MLClient.from_config(
    DefaultAzureCredential()
)

# Register the trained model
registered_model = ml_client.models.create_or_update(
    Model(
        path="data_quality_agent_model.pkl",
        name="data-quality-agent-model",
        description="Optimized Logistic Regression model for identifying data quality records requiring review.",
        type=AssetTypes.CUSTOM_MODEL
    )
)

print("Model registered successfully!")
print(f"Model name: {registered_model.name}")
print(f"Model version: {registered_model.version}")

Found the config file in: /config.json
Uploading data_quality_agent_model.pkl (< 1 MB): 100%|██████████| 3.93k/3.93k [00:00<00:00, 224kB/s]




Model registered successfully!
Model name: data-quality-agent-model
Model version: 1


In [18]:
# Create the scoring script for Azure ML deployment

score_script = """
import json
import joblib
import pandas as pd
import os

model = None

def init():
    global model
    
    model_path = os.path.join(
        os.environ["AZUREML_MODEL_DIR"],
        "data_quality_agent_model.pkl"
    )
    
    model = joblib.load(model_path)


def run(raw_data):
    try:
        data = json.loads(raw_data)
        
        if isinstance(data, dict):
            data = [data]
        
        df = pd.DataFrame(data)
        
        probabilities = model.predict_proba(df)[:, 1]
        
        threshold = 0.70
        predictions = (probabilities >= threshold).astype(int)
        
        results = []
        
        for prediction, probability in zip(predictions, probabilities):
            results.append({
                "needs_review": int(prediction),
                "review_probability": round(float(probability), 4),
                "decision": (
                    "REVIEW REQUIRED"
                    if prediction == 1
                    else "NO REVIEW REQUIRED"
                )
            })
        
        return results
        
    except Exception as e:
        return {
            "error": str(e)
        }
"""

with open("score.py", "w") as f:
    f.write(score_script)

print("Scoring script created successfully: score.py")

Scoring script created successfully: score.py


In [20]:
# Create an Azure ML online endpoint

from azure.ai.ml.entities import ManagedOnlineEndpoint

endpoint_name = "data-quality-agent-endpoint"

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Real-time endpoint for the Data Quality Agent",
    auth_mode="key"
)

ml_client.begin_create_or_update(endpoint).result()

print("Online endpoint created successfully!")
print(f"Endpoint name: {endpoint_name}")

HttpResponseError: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"\":[\"Specified endpoint [data-quality-agent-endpoint] has not been created successfully. Please recreate the endpoint.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-d3a5382cc08a3b26808c3438660cda04-00ab124482b1766a-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"\":[\"Specified endpoint [data-quality-agent-endpoint] has not been created successfully. Please recreate the endpoint.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-d3a5382cc08a3b26808c3438660cda04-00ab124482b1766a-01\"}"}}
Additional Information

In [21]:
# Delete the failed endpoint so we can recreate it cleanly

ml_client.online_endpoints.begin_delete(
    name="data-quality-agent-endpoint"
).result()

print("Failed endpoint deleted successfully!")

...Failed endpoint deleted successfully!


In [22]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

endpoint_name = "data-quality-agent-endpoint"

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Real-time endpoint for the Data Quality Agent",
    auth_mode="key"
)

ml_client.begin_create_or_update(endpoint).result()

print("Online endpoint created successfully!")
print(f"Endpoint name: {endpoint_name}")

Online endpoint created successfully!
Endpoint name: data-quality-agent-endpoint


In [23]:
import sys
import sklearn
import pandas
import joblib

print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("pandas:", pandas.__version__)
print("joblib:", joblib.__version__)

Python: 3.10.20 (main, Jun 11 2026, 15:17:37) [GCC 14.3.0]
scikit-learn: 1.7.2
pandas: 1.5.3
joblib: 1.5.3


In [24]:
# Create the environment for the Azure ML endpoint

conda_content = """
name: data-quality-agent-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - scikit-learn==1.7.2
      - pandas==1.5.3
      - joblib==1.5.3
"""

with open("conda.yml", "w") as f:
    f.write(conda_content)

print("conda.yml created successfully!")

conda.yml created successfully!


In [25]:
from azure.ai.ml.entities import Environment

deployment_env = Environment(
    name="data-quality-agent-env",
    description="Environment for the Data Quality Agent inference endpoint",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04",
    conda_file="conda.yml"
)

print("Deployment environment created successfully!")

Deployment environment created successfully!


In [28]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="data-quality-agent-endpoint",
    model="azureml:data-quality-agent-model:1",
    environment=env,
    code_configuration=CodeConfiguration(
        code=".",
        scoring_script="score.py"
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1
)

print("Starting model deployment...")

ml_client.begin_create_or_update(deployment).result()

print("Model deployment completed successfully!")

Starting model deployment...


Uploading boys3soccer (2.2 MBs): 100%|██████████| 2195267/2195267 [00:00<00:00, 14647559.71it/s]




HttpResponseError: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"VmSize\":[\"Not enough quota available for Standard_DS3_v2 in SubscriptionId fa606c55-903f-43c6-8bc7-02d6cd069bb9. Current usage/limit: 2/6. Additional needed: 8 Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-outofquota\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-60e0f60de53ea085d0559481b94809f4-dcadf48d1ab49d71-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"VmSize\":[\"Not enough quota available for Standard_DS3_v2 in SubscriptionId fa606c55-903f-43c6-8bc7-02d6cd069bb9. Current usage/limit: 2/6. Additional needed: 8 Please see troubleshooting guide, available here: https://aka.

In [29]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="data-quality-agent-endpoint",
    model="azureml:data-quality-agent-model:1",
    environment=deployment_env,
    code_configuration=CodeConfiguration(
        code=".",
        scoring_script="score.py"
    ),
    instance_type="Standard_DS2_v2",
    instance_count=1
)

print("Starting model deployment...")

ml_client.begin_create_or_update(deployment).result()

print("Model deployment completed successfully!")

Starting model deployment...
.................................................

Uploading boys3soccer (2.22 MBs): 100%|██████████| 2217232/2217232 [00:00<00:00, 11237561.37it/s]




HttpResponseError: (ResourceNotFound) The resource 'Microsoft.ContainerRegistry/registries/b78a644610b54013bc177d07db05f0aa' under resource group 'RG-Data-Quality-Agent' was not found. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotfound
Code: ResourceNotFound
Message: The resource 'Microsoft.ContainerRegistry/registries/b78a644610b54013bc177d07db05f0aa' under resource group 'RG-Data-Quality-Agent' was not found. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotfound

In [27]:
from azure.ai.ml.entities import Environment

env = Environment(
    name="data-quality-agent-env",
    description="Environment for Data Quality Agent",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04",
    conda_file="conda.yml"
)

print("Environment variable recreated successfully!")

Environment variable recreated successfully!


In [30]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(
    DefaultAzureCredential()
)

workspace = ml_client.workspaces.get("Data-Quality-Agent")

print("Workspace:", workspace.name)
print("Container Registry:", workspace.container_registry)

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Workspace: Data-Quality-Agent
Container Registry: /subscriptions/fa606c55-903f-43c6-8bc7-02d6cd069bb9/resourceGroups/RG-Data-Quality-Agent/providers/Microsoft.ContainerRegistry/registries/b78a644610b54013bc177d07db05f0aa


In [31]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="data-quality-agent-endpoint",
    model="azureml:data-quality-agent-model:1",
    environment=env,
    code_configuration=CodeConfiguration(
        code=".",
        scoring_script="score.py"
    ),
    instance_type="Standard_DS2_v2",
    instance_count=1
)

print("Starting model deployment...")

ml_client.begin_create_or_update(deployment).result()

print("Model deployment completed successfully!")

Starting model deployment...


Uploading boys3soccer (2.23 MBs): 100%|██████████| 2230831/2230831 [00:00<00:00, 11351710.24it/s]




HttpResponseError: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"\":[\"Specified deployment [blue] failed during initial provisioning and is in an unrecoverable state. Delete and re-create.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-f922a8a5599725207430c69a138c7ce8-a581d54710bc13c2-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"\":[\"Specified deployment [blue] failed during initial provisioning and is in an unrecoverable state. Delete and re-create.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-f922a8a5599725207430c69a138c7ce8-a581d54710bc13c2-01\"}"}}
Additional I

In [32]:
print("Deleting failed blue deployment...")

ml_client.online_deployments.begin_delete(
    name="blue",
    endpoint_name="data-quality-agent-endpoint"
).result()

print("Failed blue deployment deleted successfully!")

Deleting failed blue deployment...
Failed blue deployment deleted successfully!


In [33]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="data-quality-agent-endpoint",
    model="azureml:data-quality-agent-model:1",
    environment=env,
    code_configuration=CodeConfiguration(
        code=".",
        scoring_script="score.py"
    ),
    instance_type="Standard_DS2_v2",
    instance_count=1
)

print("Starting fresh model deployment...")

ml_client.begin_create_or_update(deployment).result()

print("Model deployment completed successfully!")

Starting fresh model deployment...
....................................................................................

Uploading boys3soccer (2.26 MBs): 100%|██████████| 2256602/2256602 [00:00<00:00, 22555662.57it/s]




HttpResponseError: (BadArgument) User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready
Code: BadArgument
Message: User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready

In [34]:
print("env exists:", "env" in globals())
print("deployment_env exists:", "deployment_env" in globals())

if "env" in globals():
    print("env =", env)

if "deployment_env" in globals():
    print("deployment_env =", deployment_env)

env exists: True
deployment_env exists: True
env = conda_file:
  channels:
  - conda-forge
  dependencies:
  - python=3.10
  - pip
  - pip:
    - scikit-learn==1.7.2
    - pandas==1.5.3
    - joblib==1.5.3
  name: data-quality-agent-env
description: Environment for Data Quality Agent
image: mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04
name: data-quality-agent-env
tags: {}
version: '1'

deployment_env = conda_file:
  channels:
  - conda-forge
  dependencies:
  - python=3.10
  - pip
  - pip:
    - scikit-learn==1.7.2
    - pandas==1.5.3
    - joblib==1.5.3
  name: data-quality-agent-env
description: Environment for the Data Quality Agent inference endpoint
image: mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04
name: data-quality-agent-env
tags: {}
version: '1'



In [35]:
logs = ml_client.online_deployments.get_logs(
    name="blue",
    endpoint_name="data-quality-agent-endpoint",
    lines=200
)

print(logs)

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: InProgress

Container events:
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-09-13T00:42:50.068799Z, Message: Start downloading models
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-09-13T00:42:51.61628Z, Message: Start pulling container image
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-09-13T00:43:45.455316Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-09-13T00:43:45.455316Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-09-13T00:43:45.517294Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-09-13T00:43:45.581301Z, Message: Started container inference-server

Container logs:
2026-09-13T00:43:45,605415482+00:00 - gunicorn/run 
2026-09-13T00:43:45,607181122+00:00 | gunicorn/run | 
2026-09-13T00:43:45,

In [36]:
ml_client.online_deployments.begin_delete(
    name="blue",
    endpoint_name="data-quality-agent-endpoint"
).result()

print("Failed blue deployment deleted.")

Failed blue deployment deleted.


In [37]:
conda_content = """
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - numpy
      - pandas==1.5.3
      - scikit-learn==1.7.2
      - joblib==1.5.3
      - azureml-inference-server-http
"""

with open("conda.yml", "w") as f:
    f.write(conda_content)

print("Updated conda.yml created successfully.")

Updated conda.yml created successfully.


In [38]:
from azure.ai.ml.entities import Environment

deployment_env_v2 = Environment(
    name="data-quality-agent-env-v2",
    description="Data Quality Agent inference environment with Azure ML inference server",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04",
    conda_file="conda.yml"
)

print("New deployment environment created successfully!")

New deployment environment created successfully!


In [39]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="data-quality-agent-endpoint",
    model="azureml:data-quality-agent-model:1",
    environment=deployment_env_v2,
    code_configuration=CodeConfiguration(
        code=".",
        scoring_script="score.py"
    ),
    instance_type="Standard_DS2_v2",
    instance_count=1
)

print("Starting corrected model deployment...")

ml_client.online_deployments.begin_create_or_update(
    deployment
).result()

print("Deployment completed successfully!")

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint data-quality-agent-endpoint exists
Uploading boys3soccer (2.28 MBs): 100%|██████████| 2276929/2276929 [00:00<00:00, 53288986.42it/s]




Starting corrected model deployment...
............................................................................................................................................................

HttpResponseError: (BadArgument) User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready
Code: BadArgument
Message: User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready

In [40]:
logs = ml_client.online_deployments.get_logs(
    name="blue",
    endpoint_name="data-quality-agent-endpoint",
    lines=300
)

print(logs)

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: InProgress

Container events:
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-09-13T01:13:09.262208Z, Message: Start downloading models
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-09-13T01:13:10.367069Z, Message: Start pulling container image
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-09-13T01:14:01.393187Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-09-13T01:14:01.393187Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-09-13T01:14:01.434774Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-09-13T01:14:01.49134Z, Message: Started container inference-server

Container logs:
2026-09-13T01:14:01,500258933+00:00 - rsyslog/run 
2026-09-13T01:14:01,509463616+00:00 - gunicorn/run 
2026-09-13T01:14:01,511

In [41]:
ml_client.online_deployments.begin_delete(
    name="blue",
    endpoint_name="data-quality-agent-endpoint"
).result()

print("Failed blue deployment deleted.")

Failed blue deployment deleted.


In [42]:
conda_content = """
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - numpy==1.26.4
      - pandas==1.5.3
      - scikit-learn==1.7.2
      - joblib==1.5.3
      - azureml-inference-server-http
"""

with open("conda.yml", "w") as f:
    f.write(conda_content)

print("Corrected conda.yml created successfully.")

Corrected conda.yml created successfully.


In [43]:
from azure.ai.ml.entities import Environment

deployment_env_v3 = Environment(
    name="data-quality-agent-env-v3",
    description="Data Quality Agent inference environment with compatible NumPy and Pandas versions",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04",
    conda_file="conda.yml"
)

print("Fresh deployment environment created successfully!")

Fresh deployment environment created successfully!


In [44]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="data-quality-agent-endpoint",
    model="azureml:data-quality-agent-model:1",
    environment=deployment_env_v3,
    code_configuration=CodeConfiguration(
        code=".",
        scoring_script="score.py"
    ),
    instance_type="Standard_DS2_v2",
    instance_count=1
)

print("Starting deployment with corrected environment...")

ml_client.online_deployments.begin_create_or_update(
    deployment
).result()

print("Deployment completed successfully!")

Starting deployment with corrected environment...
..............................................................................................................................................................................................Deployment completed successfully!


Uploading boys3soccer (2.3 MBs): 100%|██████████| 2302502/2302502 [00:00<00:00, 29709663.01it/s]




In [45]:
endpoint = ml_client.online_endpoints.get(
    "data-quality-agent-endpoint"
)

endpoint.traffic = {
    "blue": 100
}

ml_client.begin_create_or_update(endpoint).result()

print("Traffic routed to blue deployment.")

Traffic routed to blue deployment.


In [46]:
import json

sample_record = {
    "task_type": "Safety Review",
    "missing_value_count": 5,
    "duplicate_flag": 1,
    "format_error_flag": 1,
    "response_length": 90,
    "review_score": 60,
    "latency_seconds": 150
}

with open("sample_request.json", "w") as f:
    json.dump([sample_record], f)

response = ml_client.online_endpoints.invoke(
    endpoint_name="data-quality-agent-endpoint",
    deployment_name="blue",
    request_file="sample_request.json"
)

print(response)

[{"needs_review": 1, "review_probability": 0.993, "decision": "REVIEW REQUIRED"}]
